In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

ollama_api_key = os.getenv("OLLAMA_API_KEY")
print("Ollama api key found : ",bool(ollama_api_key))


Ollama api key found :  True


In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gpt-oss:20b-cloud",
    base_url="https://ollama.com",
    client_kwargs={
        "headers": {
            "Authorization": f"Bearer {ollama_api_key}"
        }
    },
    temperature=0
)

In [3]:
# just for checking llm
response = llm.invoke("hi")
print(response.content)


Hello! How can I help you today?


In [3]:
# create custom tool  using @tool <- decorator
from langchain_core.tools import tool

@tool
def calculate_gst(amount:float, gst_rate:float ) -> float:
    """ Calculate GST amount and final price."""
    gst = amount * gst_rate /100
    total = amount + gst 
    return {
        "amount " : amount,
        "gst_rate " : gst_rate,
        "gst " : gst,
        "Total amount " : total
    }
        
    

In [4]:
#  TOOL EXECUTION

# Yahan hum tool ko DIRECTLY execute kar rahe hain.
# LLM/Agent yahan decision nahi le raha.
#
# calculate_gst.invoke(...)
#       ↓
# Python function execute hota hai
#       ↓
# Result return hota hai

result = calculate_gst.invoke({
    "amount": 5000,
    "gst_rate": 28
})

print(result)


{'amount ': 5000.0, 'gst_rate ': 28.0, 'gst ': 1400.0, 'Total amount ': 6400.0}


In [5]:
# tool binding
llm_with_tools = llm.bind_tools([
    calculate_gst
])

# Yahan hum tool ko LLM ke saath bind kar rahe hain.
#
# Matlab:
# LLM ko bataya ja raha hai ki uske paas
# "calculate_gst" naam ka tool available hai.
#
# IMPORTANT:
# Binding ke time tool execute NAHI hota.

In [6]:
# TOOL CALLING

# Ab hum LLM ko user ka question de rahe hain.
#
# LLM decide karega:
# "Kya mujhe calculate_gst tool use karna chahiye?"
#
# Agar haan:
# LLM ek TOOL CALL generate karega.
#
# IMPORTANT:
# Is step par normally tool ka Python code execute nahi hota.
# LLM sirf tool call/request generate karta hai.

response = llm_with_tools.invoke(
    "Calculate 18% GST on ₹5000"
)

print(response)




content='' additional_kwargs={} response_metadata={'model': 'gpt-oss:20b-cloud', 'created_at': '2026-09-23T11:15:37.320454738Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1703456986, 'load_duration': None, 'prompt_eval_count': 137, 'prompt_eval_duration': None, 'eval_count': 51, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:20b-cloud', 'model_provider': 'ollama'} id='lc_run--01a0cdfa-6b61-7ec2-b2ab-16530d19f1a6-0' tool_calls=[{'name': 'calculate_gst', 'args': {'amount': 5000, 'gst_rate': 18}, 'id': 'a242e680-6022-4e53-9535-5e24b27bfa22', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 137, 'output_tokens': 51, 'total_tokens': 188}


In [7]:
# ============================================================
# TOOL CALL INFORMATION
# ============================================================
# Agar model tool call generate karta hai,
# to hum tool_calls dekh sakte hain.

print(response.tool_calls)

[{'name': 'calculate_gst', 'args': {'amount': 5000, 'gst_rate': 18}, 'id': 'a242e680-6022-4e53-9535-5e24b27bfa22', 'type': 'tool_call'}]


In [8]:
# ================================
# TOOL EXECUTION
# ================================

tool_call = response.tool_calls[0]

result = calculate_gst.invoke(tool_call["args"])

print(result)

{'amount ': 5000.0, 'gst_rate ': 18.0, 'gst ': 900.0, 'Total amount ': 5900.0}
